# 03 — Environment & Metrics

| Module | Contents |
|---|---|
| `environment/cyber_env.py` | Gymnasium env bridging attacker and defender |
| `evaluation/metrics.py` | Step/episode recording and visualization |

## State space (24 features)
```
[0:10]   attack_type one-hot        (10 classes)
[10:17]  kill_chain_stage one-hot   (7 stages)
[17]     threat_level               float [0, 1]
[18]     attack_count_normalized    float [0, 1]
[19]     escalation_rate            float [0, 1]
[20:24]  attacker_intent one-hot    (4 intents)
```

## Action space (discrete, 5)
```
0 = ALLOW   1 = LOG   2 = TROLL   3 = BLOCK   4 = ALERT
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

---
## 1. Cybersecurity Environment (`cyber_env.py`)

In [ ]:
from environment.cyber_env import CyberSecurityEnv
from attacker.attack_types import AttackerIntent

env = CyberSecurityEnv(
    attacker_intent=AttackerIntent.OPPORTUNISTIC,
    max_steps=200,
    seed=42,
)

print('Observation space :', env.observation_space)
print('Action space      :', env.action_space)

In [ ]:
# Inspect the initial state vector
state, info = env.reset()
print('State shape :', state.shape)
print('State dtype :', state.dtype)
print('\nFirst info dict:')
for k, v in info.items():
    print(f'  {k:22s} = {v}')

### 1.1 Run a random-policy episode and record results

In [ ]:
from attacker.attack_types import AttackType, KillChainStage
from defender.honeypot import HoneypotAction

env = CyberSecurityEnv(
    attacker_intent=AttackerIntent.AGGRESSIVE,
    max_steps=150,
    seed=0,
)
state, info = env.reset()

trajectory = []
done = False
while not done:
    action = env.action_space.sample()   # random policy
    next_state, reward, terminated, truncated, info = env.step(action)
    trajectory.append({
        'step':         info['step'],
        'attack_type':  info['attack_type'].name,
        'stage':        info['kill_chain_stage'].name,
        'threat':       info['threat_level'],
        'escalation':   info['escalation_rate'],
        'action':       HoneypotAction(action).name,
        'reward':       reward,
        'is_attack':    info['is_attack'],
    })
    state = next_state
    done = terminated or truncated

traj_df = pd.DataFrame(trajectory)
print(f'Episode length  : {len(traj_df)} steps')
print(f'Total reward    : {traj_df["reward"].sum():.2f}')
print(f'Attack fraction : {traj_df["is_attack"].mean():.2%}')
traj_df.head(8)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Random-Policy Episode — AGGRESSIVE attacker', fontsize=13, fontweight='bold')

# Threat level
axes[0].fill_between(traj_df.step, traj_df.threat, alpha=0.3, color='red')
axes[0].plot(traj_df.step, traj_df.threat, color='red', linewidth=1.5)
for th, label, c in [(0.75,'critical','darkred'),(0.55,'high','orange'),(0.35,'medium','gold')]:
    axes[0].axhline(th, linestyle='--', color=c, alpha=0.6, label=label)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Threat Level')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].grid(True, alpha=0.3)

# Kill chain stage
stage_num = traj_df.stage.map({s: i for i, s in enumerate(KillChainStage.names())})
axes[1].step(traj_df.step, stage_num, where='post', color='navy')
axes[1].set_yticks(range(KillChainStage.count()))
axes[1].set_yticklabels(KillChainStage.names(), fontsize=8)
axes[1].set_ylabel('Kill Chain Stage')
axes[1].grid(True, alpha=0.3)

# Reward
color_r = ['green' if r >= 0 else 'red' for r in traj_df.reward]
axes[2].bar(traj_df.step, traj_df.reward, color=color_r, alpha=0.7, width=1)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_ylabel('Reward')
axes[2].set_xlabel('Step')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Metrics Collector (`metrics.py`)

`MetricsCollector` records per-step data, aggregates it into per-episode summaries
(`EpisodeRecord`), and provides plotting utilities.

In [ ]:
from evaluation.metrics import MetricsCollector, StepRecord

# Run 10 random episodes and collect metrics
env = CyberSecurityEnv(
    attacker_intent=AttackerIntent.OPPORTUNISTIC,
    max_steps=200,
    seed=1,
)
metrics = MetricsCollector(log_dir='../logs/notebook_demo/')

for ep in range(10):
    state, info = env.reset()
    done = False
    while not done:
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, info = env.step(action)
        pred_attack = AttackType.NORMAL   # placeholder (no classifier)
        metrics.record_step(ep, info['step'], action, reward, info, pred_attack, loss=None)
        state = next_state
        done = terminated or truncated
    metrics.end_episode(ep)

summary = metrics.summary_report()
print('Summary across 10 random-policy episodes:')
for k, v in summary.items():
    print(f'  {k:<30s}: {v:.4f}' if isinstance(v, float) else f'  {k:<30s}: {v}')

In [ ]:
# Episode-level summary table
ep_df = pd.DataFrame([
    {
        'episode':             e.episode,
        'total_reward':        e.total_reward,
        'steps':               e.steps,
        'detection_rate':      e.detection_rate,
        'false_positive_rate': e.false_positive_rate,
        'avg_threat_level':    e.avg_threat_level,
    }
    for e in metrics.episodes
])
ep_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Episode Metrics — Random Policy (10 episodes)', fontsize=12)

ep_df.plot(x='episode', y='total_reward', ax=axes[0], legend=False, color='steelblue', marker='o')
axes[0].set_title('Total Reward')
axes[0].set_ylabel('Reward')

ep_df.plot(x='episode', y='detection_rate', ax=axes[1], legend=False, color='green', marker='o')
ep_df.plot(x='episode', y='false_positive_rate', ax=axes[1], legend=False, color='red',
           marker='x', linestyle='--')
axes[1].set_title('Detection Rate (green) vs FP Rate (red)')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Rate')

ep_df.plot(x='episode', y='avg_threat_level', ax=axes[2], legend=False, color='orange', marker='o')
axes[2].set_title('Avg Threat Level')
axes[2].set_ylim(0, 1)
axes[2].set_ylabel('Threat Level')

plt.tight_layout()
plt.show()

### 2.1 Action distribution across episodes

In [ ]:
action_counts = {name: 0 for name in HoneypotAction.names()}
for ep in metrics.episodes:
    for name, cnt in ep.action_dist.items():
        action_counts[name] = action_counts.get(name, 0) + cnt

act_series = pd.Series(action_counts)
act_series.plot(kind='bar', color='steelblue', figsize=(7, 3))
plt.title('Total action counts across 10 episodes (random policy)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()